# 📄 Smart CV Analyzer: Tahap 1 - Ekstraksi PDF ke CSV

Notebook ini digunakan untuk mengekstrak teks dari kumpulan file CV berformat PDF, melakukan *preprocessing* dasar, dan memecahnya menjadi per kalimat sebelum diproses lebih lanjut untuk anotasi BIO Tagging.

Target output dari notebook ini adalah sebuah file CSV yang berisi kolom `id` dan `text`.

## 1. Import Library
Menggunakan `pdfplumber` untuk menjaga struktur teks dan tata letak pada file PDF sehingga kata-kata tidak mudah menyatu berantakan jika CV memiliki format dua kolom.

In [1]:
import os
import pdfplumber
import pandas as pd
import re

## 2. Ekstraksi dan Pemecahan Kalimat (*Sentence Splitting*)
Sistem akan me-*looping* seluruh file PDF di dalam folder. Teks yang berhasil diekstrak akan dipecah berdasarkan tanda titik `.` atau baris baru `\n`. Teks yang terlalu pendek (seperti nomor halaman atau poin kosong) akan dibuang secara otomatis.

In [2]:
# Tentukan lokasi folder tempat 50 file PDF CV berada
folder_path = "cv_dummy_50_IT" 

dataset_kalimat = []
sentence_id = 1

print(f"Mencari file PDF di folder: '{folder_path}'...\n")

# Looping untuk membaca semua file PDF di dalam folder
for filename in os.listdir(folder_path):
    if filename.endswith(".pdf"):
        file_path = os.path.join(folder_path, filename)
        
        try:
            with pdfplumber.open(file_path) as pdf:
                text = ""
                # Ambil teks dari setiap halaman CV
                for page in pdf.pages:
                    extracted = page.extract_text()
                    if extracted:
                        text += extracted + "\n"
                
                # Preprocessing & Pemecahan Kalimat
                # Pecah teks berdasarkan titik diikuti spasi (. ) ATAU baris baru (\n)
                raw_sentences = re.split(r'(?:\.\s|\n)', text)
                
                for s in raw_sentences:
                    # Bersihkan spasi ganda, tab, dan karakter aneh
                    clean_sentence = re.sub(r'\s+', ' ', s).strip()
                    
                    # Hanya ambil kalimat yang panjangnya lebih dari 15 karakter
                    if len(clean_sentence) > 15:
                        dataset_kalimat.append({
                            "id": f"{sentence_id:04d}",
                            "text": clean_sentence
                        })
                        sentence_id += 1
                        
        except Exception as e:
            print(f"❌ Gagal membaca file {filename}: {e}")

print("✅ Ekstraksi teks selesai!")

Mencari file PDF di folder: 'cv_dummy_50_IT'...

✅ Ekstraksi teks selesai!


### 3. Konversi ke DataFrame dan Ekspor ke CSV
Mengubah datanya menjadi tabel (DataFrame) pandas dan menyimpannya ke dalam file CSV agar siap digunakan untuk tahap selanjutnya (Tokenisasi & BIO Tagging).

In [3]:
# Simpan kumpulan kalimat ke DataFrame
df_cv = pd.DataFrame(dataset_kalimat)

# Ekspor ke file CSV
csv_filename = "cv_sentences_raw.csv"
df_cv.to_csv(csv_filename, index=False)

print(f"✅ Data berhasil disimpan ke: {csv_filename}")
print(f"📊 Total kalimat yang berhasil diekstrak: {len(df_cv)} kalimat")
print("\nPreview Data (5 baris pertama):")
display(df_cv.head())

✅ Data berhasil disimpan ke: cv_sentences_raw.csv
📊 Total kalimat yang berhasil diekstrak: 1179 kalimat

Preview Data (5 baris pertama):


,id,text
0,0001,Jakarta | +62 845-5012-4657 | pandu.wijaya18@g...
1,0002,RINGKASAN PROFIL
2,0003,Data engineer yang berpengalaman dalam membang...
3,0004,bekerja dengan berbagai teknologi big data unt...
4,0005,Memiliki kemampuan
